# Deploy Qwen3-VL Vision-Language Models on Amazon SageMaker

This notebook demonstrates how to deploy **ALL** Qwen3-VL vision-language models on Amazon SageMaker using a custom vLLM container.

## Supported Models (Choose One)

### Qwen3-VL-235B-A22B-Instruct
- **Parameters**: 236B (Mixture of Experts)
- **Instance Type**: ml.p5.48xlarge
- **GPUs**: 8x NVIDIA H100 (640GB VRAM total)
- **Highlights**: Maximum capability for research and state-of-the-art performance. Only when absolute best accuracy is required.

### Qwen3-VL-30B-A3B-Instruct
- **Parameters**: 31B (Mixture of Experts)
- **Instance Type**: ml.g6.12xlarge
- **GPUs**: 4x NVIDIA L4 (96GB VRAM)
- **Highlights**: High accuracy for complex visual reasoning. Medical imaging, scientific research, advanced OCR.

### Qwen3-VL-8B-Instruct (RECOMMENDED)
- **Parameters**: 9B (Dense)
- **Instance Type**: ml.g6e.xlarge
- **GPUs**: 1x NVIDIA L40S (48GB VRAM)
- **Highlights**: Best balance of performance and cost. Ideal for production applications. Proven reliability.

### Qwen3-VL-4B-Instruct
- **Parameters**: 4B (Dense)
- **Instance Type**: ml.g5.xlarge
- **GPUs**: 1x NVIDIA L4 (24GB VRAM)
- **Highlights**: Perfect for development, testing, and cost-sensitive applications. Fastest iteration speed.


## Model Capabilities
- Advanced spatial perception (2D/3D reasoning)
- Multi-language OCR (32 languages)
- Visual agent functionality
- Video understanding with timestamps
- Visual coding generation
- Context Length: 256K tokens (expandable to 1M)

## Prerequisites

1. AWS CLI configured with appropriate permissions
2. SageMaker execution role with necessary permissions
3. Access to appropriate SageMaker instance types
4. Custom Docker image pushed to ECR using `build_and_push.sh` script

In [1]:
# Install required packages
!pip install -U boto3 sagemaker -q

In [ ]:
import boto3
import sagemaker
from sagemaker.model import Model
from sagemaker.predictor import Predictor
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer
import json
import time
from datetime import datetime
import base64
from typing import Dict, List, Any, Optional

# Initialize SageMaker session and get execution role
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = "3p-projects"

# Get AWS account and region information
sts_client = boto3.client('sts')
account_id = sts_client.get_caller_identity()['Account']
region = boto3.Session().region_name

print(f"AWS Account ID: {account_id}")
print(f"AWS Region: {region}")
print(f"SageMaker role: {role}")
print(f"S3 bucket: {bucket}")

In [5]:
# Configuration for custom container
repository_name = "qwen3-vl-byoc"
image_tag = "latest"
image_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{repository_name}:{image_tag}"

print(f"Custom container image:")
print(f"Repository: {repository_name}")
print(f"Image URI: {image_uri}")

Custom container image:
Repository: qwen3-vl-byoc
Image URI: 459006231907.dkr.ecr.us-west-2.amazonaws.com/qwen3-vl-byoc:latest


## Model Selection - Choose Your Model

Select which Qwen3-VL model to deploy based on your requirements

In [35]:
# 🎯 Model Selection - CHOOSE ONE
MODEL_CHOICE = "30B"  # Options: "4B", "8B", "30B", "235B"

# Model configurations, change if needed

MODEL_CONFIGS = {
    "4B": {
        "model_id": "Qwen/Qwen3-VL-4B-Instruct",
        "model_name": "qwen3-vl-4b-instruct",
        "instance_type": "ml.g6.xlarge",
        "tensor_parallel_size": 1,
        "max_model_len": 32768,
        "gpu_memory_util": 0.9,
    },
    "8B": {
        "model_id": "Qwen/Qwen3-VL-8B-Instruct",
        "model_name": "qwen3-vl-8b-instruct",
        "instance_type": "ml.g6e.xlarge",
        "tensor_parallel_size": 1,
        "max_model_len": 32768,
        "gpu_memory_util": 0.9,
    },
    "30B": {
        "model_id": "Qwen/Qwen3-VL-30B-A3B-Instruct",
        "model_name": "qwen3-vl-30b-a3b-instruct",
        "instance_type": "ml.g6.12xlarge",
        "tensor_parallel_size": 4,
        "max_model_len": 32768,
        "gpu_memory_util": 0.9,
    },
    "235B": {
        "model_id": "Qwen/Qwen3-VL-235B-A22B-Instruct",
        "model_name": "qwen3-vl-235b-a22b-instruct",
        "instance_type": "ml.p5.48xlarge",
        "tensor_parallel_size": 8,
        "max_model_len": 32768,
        "gpu_memory_util": 0.85,
    }
}

if MODEL_CHOICE not in MODEL_CONFIGS:
    raise ValueError(f"MODEL_CHOICE must be one of: {list(MODEL_CONFIGS.keys())}")

config = MODEL_CONFIGS[MODEL_CHOICE]
model_id = config["model_id"]
model_name = config["model_name"]
instance_type = config["instance_type"]
tensor_parallel_size = config["tensor_parallel_size"]
max_model_len = config["max_model_len"]
gpu_memory_util = config["gpu_memory_util"]

# Generate unique endpoint name
endpoint_name = f"{model_name}-{int(time.time())}"
instance_count = 1
prefix = "qwen3-vl-deployment"

print(f"🎯 Selected: {MODEL_CHOICE}")

print(f"\n🚀 Deployment Configuration:")
print(f"   Model ID: {model_id}")
print(f"   Model Name: {model_name}")
print(f"   Endpoint Name: {endpoint_name}")
print(f"   S3 Bucket: {bucket}")
print(f"   S3 Prefix: {prefix}")

🎯 Selected: 30B

🚀 Deployment Configuration:
   Model ID: Qwen/Qwen3-VL-30B-A3B-Instruct
   Model Name: qwen3-vl-30b-a3b-instruct
   Endpoint Name: qwen3-vl-30b-a3b-instruct-1760710642
   S3 Bucket: 3p-projects
   S3 Prefix: qwen3-vl-deployment


## Step 1: Create SageMaker Model Configuration

Now we'll create the SageMaker model configuration with optimized settings for your selected model.

In [36]:
# Create SageMaker model configuration
print("🏗️ Creating SageMaker model configuration...")

# Build CUDA_VISIBLE_DEVICES string
cuda_devices = ','.join(map(str, range(tensor_parallel_size)))

model = Model(
    image_uri=image_uri,
    model_data=None,
    role=role,
    name=f"{model_name}-model-{int(time.time())}",
    sagemaker_session=sagemaker_session,
    env={
        # 🚀 vLLM Configuration
        'VLLM_USE_V1': '1',
        'VLLM_WORKER_MULTIPROC_METHOD': 'spawn',
        'VLLM_DISTRIBUTED_EXECUTOR_BACKEND': 'mp' if tensor_parallel_size > 1 else 'ray',
        'VLLM_LOGGING_LEVEL': 'INFO',
        
        # 🗂️ Cache Directories
        'MODEL_CACHE_DIR': '/opt/ml/model',
        'TRANSFORMERS_CACHE': '/tmp/transformers_cache',
        'HF_HOME': '/tmp/hf_home',
        
        # 🌐 Server Configuration
        'SAGEMAKER_BIND_TO_PORT': '8080',
        'SAGEMAKER_BIND_TO_HOST': '0.0.0.0',
        
        # 🔧 Model-Specific Settings
        'OPTION_MODEL': model_id,
        'OPTION_TENSOR_PARALLEL_SIZE': str(tensor_parallel_size),
        'OPTION_MAX_MODEL_LEN': str(max_model_len),
        'OPTION_GPU_MEMORY_UTILIZATION': str(gpu_memory_util),
        'OPTION_DTYPE': 'auto',
        'OPTION_TRUST_REMOTE_CODE': 'true',
        'OPTION_MAX_NUM_SEQS': '32' if tensor_parallel_size == 1 else '64',
        'OPTION_SERVED_MODEL_NAME': model_name,
        
        # 🖼️ Vision-Specific Settings
        'OPTION_LIMIT_MM_PER_PROMPT': '{"image":10,"video":1}',
        'OPTION_MAX_NUM_BATCHED_TOKENS': '8192',
        
        # 🛠️ Tool Calling Support
        'OPTION_ENABLE_AUTO_TOOL_CHOICE': 'true',
        'OPTION_TOOL_CALL_PARSER': 'hermes',
        
        # 🔧 Optimization Settings
        'NCCL_DEBUG': 'INFO' if tensor_parallel_size > 1 else 'WARN',
        'TORCH_COMPILE_DISABLE': '1',
        'VLLM_DISABLE_CUSTOM_ALL_REDUCE': '1' if tensor_parallel_size > 1 else '0',
    }
)

print(f"✅ SageMaker model created successfully!")
print(f"   Model will use {tensor_parallel_size} GPU(s)")

🏗️ Creating SageMaker model configuration...
✅ SageMaker model created successfully!
   Model will use 4 GPU(s)


## Step 2: Deploy Model to SageMaker Endpoint

This step creates and deploys the model to a real-time inference endpoint.

**⏱️ Expected Time**: 10-20 minutes (longer for larger models)

In [37]:
# Deploy the model
print(f"🚀 Starting deployment to endpoint: {endpoint_name}")
print(f"⏱️  Estimated Time: 10-20 minutes (depending on model size)")

# Adjust timeout based on model size
timeout = 1200 if MODEL_CHOICE in ["4B", "8B"] else 1800

predictor = model.deploy(
    initial_instance_count=instance_count,
    instance_type=instance_type,
    endpoint_name=endpoint_name,
    serializer=JSONSerializer(),
    deserializer=JSONDeserializer(),
    container_startup_health_check_timeout=timeout,
    model_data_download_timeout=timeout,
    wait=True,
)

print(f"✅ Deployment completed successfully!")

🚀 Starting deployment to endpoint: qwen3-vl-30b-a3b-instruct-1760710642
⏱️  Estimated Time: 10-20 minutes (depending on model size)
--------------!✅ Deployment completed successfully!


In [38]:
# Connect to endpoint
predictor = Predictor(
    endpoint_name= endpoint_name,
    serializer=JSONSerializer(),
    deserializer=JSONDeserializer()
)
print(f"🔗 Connected to endpoint: {predictor.endpoint_name}")

🔗 Connected to endpoint: qwen3-vl-30b-a3b-instruct-1760710642


## Step 3: Test Inference

Test the deployed model with various scenarios.

In [39]:
# Test 1: Text-only
from IPython.display import display, Markdown

request = {
    "messages": [
        {
            "role": "user",
            "content": "Hello! Can you tell me what you can do?"
        }
    ],
    "max_tokens": 200,
    "temperature": 0.7
}

start = time.time()
response = predictor.predict(request)
end = time.time()

print(f"✅ Response time: {end-start:.2f}s")
display(Markdown(response["choices"][0]["message"]["content"]))

✅ Response time: 5.64s


Hello! 😊 I'm glad you're curious about what I can do! I'm a large-scale language model capable of assisting with a wide variety of tasks. Here are some of the things I can help you with:

### 1. **Answer Questions**
- Whether it's general knowledge, science, history, culture, or current events, I can provide informative and accurate answers.
- I can explain complex topics in simple terms.

### 2. **Write Content**
- I can help you write stories, official documents, emails, scripts, and more.
- I can create creative content like poems, scripts, or even song lyrics.

### 3. **Logical Reasoning & Problem Solving**
- I can assist with math problems, coding, logic puzzles, and analytical tasks.
- I can help debug code or suggest improvements.

### 4. **Programming Assistance**
- I can write, explain, or debug code in various programming languages (e.g., Python, JavaScript,

In [40]:
# 🧪 Test 2: Image Understanding
print("🖼️ Testing Image Understanding...")

image_request = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "What do you see in this image? Describe it in detail."
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg"
                    }
                }
            ]
        }
    ],
    "max_tokens": 500,
    "temperature": 0.7
}

# Measure inference time
start_time = time.time()
response = predictor.predict(image_request)
end_time = time.time()

# Display results
print(f"✅ Image Understanding Test Completed")
print(f"   Response Time: {end_time - start_time:.2f} seconds")
print()

# Render the response
display(Markdown("**Image Analysis:**"))
display(Markdown(response["choices"][0]["message"]["content"]))

🖼️ Testing Image Understanding...
✅ Image Understanding Test Completed
   Response Time: 10.62 seconds



**Image Analysis:**

This is a detailed description of the image provided:

The image captures a serene and expansive natural landscape, likely a marsh or wetland, viewed from a low angle that emphasizes the depth of the scene. The central focus is a weathered wooden boardwalk that leads the viewer's eye from the immediate foreground into the distance. The boardwalk is constructed from light-colored wooden planks and is flanked by thick, vibrant green grass and reeds that are taller than the walkway itself.

The landscape is dominated by a vast, flat expanse of lush green vegetation, suggesting a healthy, wet environment. On the right side of the frame, the grasses appear slightly more golden and dense, while on the left, they are a brighter, more uniform green. In the far background, a line of trees and shrubs forms a natural horizon, separating the field from the sky.

The sky above is a brilliant, deep blue, filled with delicate, wispy white clouds that are stretched horizontally, possibly indicating a high altitude or wind at that level. The lighting suggests it is either early morning or late afternoon, as the sun is low, casting a warm, golden glow that illuminates the tops of the grasses and creates a soft, inviting atmosphere. The overall composition is balanced and peaceful, evoking a sense of calm and the beauty of nature.

In [41]:
# 🧪 Test 3: OCR Capabilities
print("📊 Testing OCR Capabilities...")

import base64
from pathlib import Path

# Function to encode image to base64
def encode_image_to_base64(image_path: str) -> str:
    """Encode an image file to base64 string."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')


# Encode the image
image_path = "invoice.png"
if Path(image_path).exists():
    base64_image = encode_image_to_base64(image_path)
    
    local_image_request = {
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "Extract all the text you can read from this image, and generate response in JSON format"
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{base64_image}"
                        }
                    }
                ]
            }
        ],
        "max_tokens": 1000,
        "temperature": 0.0
    }
    
    # Measure inference time
    start_time = time.time()
    response = predictor.predict(local_image_request)
    end_time = time.time()
    
    # Display results
    print(f"✅ OCR Test Completed")
    print(f"   Response Time: {end_time - start_time:.2f} seconds")
    print()
    
    # Print the response
    print(response["choices"][0]["message"]["content"])
else:
    print("⚠️ Image file not found. Please provide a valid image path.")

📊 Testing OCR Capabilities...
✅ OCR Test Completed
   Response Time: 7.57 seconds

```json
{
  "invoice": {
    "title": "INVOICE",
    "issued_to": {
      "name": "Richard Sanchez",
      "company": "Thynk Unlimited",
      "address": "123 Anywhere St., Any City"
    },
    "pay_to": {
      "bank": "Borcele Bank",
      "account_name": "Adeline Palmerston",
      "account_number": "0123 4567 8901"
    },
    "invoice_details": {
      "invoice_no": "01234",
      "date": "11.02.2030",
      "due_date": "11.03.2030"
    },
    "items": [
      {
        "description": "Brand consultation",
        "unit_price": 100,
        "qty": 1,
        "total": 100
      },
      {
        "description": "logo design",
        "unit_price": 100,
        "qty": 1,
        "total": 100
      },
      {
        "description": "Website design",
        "unit_price": 100,
        "qty": 1,
        "total": 100
      },
      {
        "description": "Social media templates",
        "unit_price": 10

In [42]:
# 🧪 Test 4: Spatial Reasoning with Multiple Images
print("🎨 Testing Spatial Reasoning...")

spatial_request = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Compare these two images and describe the differences you notice."
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://upload.wikimedia.org/wikipedia/commons/9/97/Polar_bear_cubs_in_the_snow.jpg"
                    }
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://upload.wikimedia.org/wikipedia/commons/2/2a/Brown_bear_%28Ursus_arctos_arctos%29_running.jpg"
                    }
                }
            ]
        }
    ],
    "max_tokens": 800,
    "temperature": 0.7
}

# Measure inference time
start_time = time.time()
response = predictor.predict(spatial_request)
end_time = time.time()

# Display results
print(f"✅ Spatial Reasoning Test Completed")
print(f"   Response Time: {end_time - start_time:.2f} seconds")
print()

# Render the response
display(Markdown("**Comparison Analysis:**"))
display(Markdown(response["choices"][0]["message"]["content"]))

🎨 Testing Spatial Reasoning...
✅ Spatial Reasoning Test Completed
   Response Time: 10.63 seconds



**Comparison Analysis:**

The two images depict two different species of bears in distinct environments and life stages.
 Here is a detailed comparison of the differences:
- **Species and Age:**  
- The first image features two young polar bear cubs.
 They are identifiable by their pure white, thick fur, which provides camouflage in their snowy habitat.
  
- The second image shows a large, adult brown bear (likely a grizzly).
 Its fur is a brownish-tan color and appears wet, with a prominent shoulder hump, a characteristic feature of grizzly bears.

- **Setting and Environment:**  
- The polar bear cubs are inside a snow den, a structure they create for shelter and raising their young in the Arctic.
 The environment is entirely white and cold.
  
- The brown bear is in a terrestrial, natural habitat, running through green grass near a body of water.
 The setting appears temperate or subarctic, not arctic.

- **Behavior and Activity:**  
- The polar bear cubs are resting and interacting.
 One cub is sitting up, while the other is lying down and nuzzling its sibling, suggesting a moment of calm and bonding.
  
- The brown bear is in motion, captured mid-stride as it runs.
 Its mouth is open, and water is spraying from its fur, indicating it is active and possibly just emerged from the water.

- **Physical Characteristics:**  
- The cubs are small, with round faces and relatively short snouts, typical of young animals.
  
- The adult brown bear is large and muscular, with a longer snout and a more robust build.

---
In summary, the primary differences are the species (polar bear cubs vs. a brown bear), their age (young vs. adult), their environment (a snowy den vs. a grassy area near water), and their activity (resting vs. running).

## Step 4: Cleanup

**⚠️ Important**: Delete endpoint when done to stop billing.

In [ ]:
# delete endpoint
predictor.delete_endpoint(delete_endpoint_config=True)
print(f"✅ Endpoint {endpoint_name} deleted")



## Summary

### ✅ What We Accomplished

1. **Successfully deployed** Qwen3-VL vision-language model on SageMaker
2. **Configured vLLM** with optimized settings for vision capabilities
3. **Tested inference** with multiple scenarios:
   - Text-only conversations
   - Image understanding from URLs
   - OCR and text extraction
   - Spatial reasoning with multiple images